# 005 · Long flux pulse on one SHFQC — solution

**Level:** basic · **Topics:** Gen2, SHFQC, flux, calibration, pulses, markers, sweeps · **Type:** write the code

Solution of [exercise.ipynb](exercise.ipynb).

In [ ]:
import numpy as np

from laboneq.contrib.example_helpers.plotting.plot_helpers import plot_simulation
from laboneq.simple import *

## Device setup

One SHFQC+. The flux line uses the second signal generator channel.

📖 [Device setup](https://docs.zhinst.com/labone_q_user_manual/core/functionality_and_concepts/00_device_setup/concepts/index.html)

In [ ]:
setup = DeviceSetup("exercise_setup")
setup.add_dataserver(host="localhost", port="8004")
setup.add_instruments(
    SHFQC(uid="shfqc", address="DEV12001", device_options="SHFQC/PLUS/QC2CH")
)
setup.add_connections(
    "shfqc",
    create_connection(to_signal="q0/drive", ports="SGCHANNELS/0/OUTPUT"),
    create_connection(to_signal="q0/flux", ports="SGCHANNELS/1/OUTPUT"),
    create_connection(to_signal="q0/measure", ports="QACHANNELS/0/OUTPUT"),
    create_connection(to_signal="q0/acquire", ports="QACHANNELS/0/INPUT"),
)

## Step 1: calibration

The drive is a microwave signal at 5.1 GHz. The flux pulse is a baseband (DC) pulse:
the signal generator channel must use its **LF path**, with a centre frequency of 0 Hz.

📖 [Calibrating signals](https://docs.zhinst.com/labone_q_user_manual/core/functionality_and_concepts/02_logical_signals/concepts/00_calibrating_signals.html) ·
[Calibration properties](https://docs.zhinst.com/labone_q_user_manual/core/functionality_and_concepts/02_logical_signals/concepts/02_calibration_properties.html)

In [ ]:
calibration = Calibration()
calibration["q0/drive"] = SignalCalibration(
    oscillator=Oscillator(frequency=100e6, modulation_type=ModulationType.HARDWARE),
    local_oscillator=Oscillator(frequency=5.0e9),
    range=0,
)
calibration["q0/flux"] = SignalCalibration(
    port_mode=PortMode.LF,
    local_oscillator=Oscillator(frequency=0),
    range=0,
)
calibration["q0/measure"] = SignalCalibration(
    oscillator=Oscillator(frequency=50e6),
    local_oscillator=Oscillator(frequency=7.0e9),
    range=0,
)
calibration["q0/acquire"] = SignalCalibration(
    oscillator=Oscillator(frequency=50e6),
    local_oscillator=Oscillator(frequency=7.0e9),
    range=0,
)
setup.set_calibration(calibration)

## Session

📖 [Session](https://docs.zhinst.com/labone_q_user_manual/core/functionality_and_concepts/01_session/concepts/index.html)

In [ ]:
session = Session(setup)
session.connect(do_emulation=True)

## Step 2: the sequence

📖 [Pulse commands](https://docs.zhinst.com/labone_q_user_manual/core/functionality_and_concepts/03_sections_pulses/concepts/00_pulse_commands.html) ·
[Triggers and markers](https://docs.zhinst.com/labone_q_user_manual/core/functionality_and_concepts/03_sections_pulses/concepts/07_trigger_marker.html) ·
[Amplitude and phase of pulses](https://docs.zhinst.com/labone_q_user_manual/core/functionality_and_concepts/03_sections_pulses/concepts/06_pulses_amplitude_and_phase.html) ·
[Averaging and sweeping](https://docs.zhinst.com/labone_q_user_manual/core/functionality_and_concepts/03_sections_pulses/concepts/04_averaging_sweeping.html)

In [ ]:
flux_pulse = pulse_library.const(length=50e-6, amplitude=0.5, can_compress=True)
drive_pulse = pulse_library.gaussian(length=100e-9, amplitude=1.0)
readout_pulse = pulse_library.const(length=1e-6, amplitude=0.5)

flux_length = 50e-6
marker_length = 100e-9

amplitudes = LinearSweepParameter(start=0.1, stop=1.0, count=21)
phases = LinearSweepParameter(start=0, stop=2 * np.pi, count=21)

In [ ]:
@dsl.experiment(signals=["drive", "flux", "measure", "acquire"])
def flux_sequence():
    with dsl.acquire_loop_rt(count=16):
        with dsl.sweep(parameter=[amplitudes, phases]) as (amplitude, phase):
            with dsl.section(name="flux") as flux_section:
                dsl.play("flux", flux_pulse)
                dsl.play(
                    "drive",
                    pulse=None,
                    marker={"marker1": {"start": 0, "length": marker_length}},
                )
                dsl.delay("drive", time=flux_length - marker_length - 200e-9)
                dsl.play("drive", drive_pulse, amplitude=amplitude)
                dsl.play("drive", drive_pulse, phase=phase)
            with dsl.section(name="readout", play_after=flux_section):
                dsl.measure(
                    measure_signal="measure",
                    acquire_signal="acquire",
                    measure_pulse=readout_pulse,
                    integration_kernel=readout_pulse,
                    handle="result",
                    reset_delay=1e-6,
                )


experiment = flux_sequence()
experiment.set_signal_map(
    {
        "drive": "q0/drive",
        "flux": "q0/flux",
        "measure": "q0/measure",
        "acquire": "q0/acquire",
    }
)

## Step 3: compile and run

With `can_compress=True` the flat part of the flux pulse is not stored sample by sample, so it fits.

In [ ]:
compiled = session.compile(experiment)
results = session.run(compiled)

In [ ]:
# check
print(results["result"].data.shape)

**Expected result:** `(21,)`

Look at the end of the first shot. The two drive pulses must sit at the end of the flux pulse, and the readout must start when the flux pulse ends (50 µs).

In [ ]:
plot_simulation(
    compiled, start_time=49.5e-6, length=2e-6, signals=["flux", "drive", "measure"]
)

## Answers

1. It fails when the experiment runs: `Local oscillator frequency mismatch between outputs 0 and 1 sharing synthesizer`. On the SHFQC, signal generator channels share synthesizers in pairs. In LF mode the channel does not use the synthesizer at all, and `local_oscillator` only sets the digital centre frequency, here 0 Hz for a DC pulse.
2. No. The flux pulse is not swept: it is one single waveform of 50 µs, and it is this one waveform that is too long. Chunking only helps when there are too many *different* waveforms.
3. Amplitude and phase are applied in real time to one stored waveform. The drive needs only one 100 ns waveform for all sweep points.
4. Pulses with a constant part, such as `const`. The instrument plays the first samples, holds the value, then plays the last samples. A `gaussian` has no constant part, so `can_compress` cannot help there.

**Takeaway:** one SHFQC is enough for drive, readout and flux, if the flux channel is in LF mode and the long pulse is compressed.